# CogMem Phase 2 — Q-Weighted LoRA Training (Qwen2.5:3b)\n\nTrain LoRA on Qwen2.5:3b using the model's **own meta-reflections** weighted by Q-values.\nNo expert data — pure self-supervised improvement.\n\n**Requires:** memory_bank_qwen.json from Phase 1 (upload to /notebooks/CogMem/results/)

In [ ]:
# Cell 1: Install deps (DO NOT touch torch)
!pip install "transformers==4.43.4" "peft==0.13.2" "accelerate==0.33.0" "bitsandbytes==0.43.3" "datasets==2.20.0" "huggingface-hub==0.24.0" "pydantic>=2.0" pyyaml -q
!python3 -c "import torch; print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"
!python3 -c "from transformers import Trainer; print('Trainer OK')"
print("Restart kernel, run Cell 2")

In [ ]:
# Cell 2: Verify + HuggingFace login
import torch, transformers, peft
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from huggingface_hub import login
login(token="PUT_YOUR_TOKEN_HERE")

In [ ]:
# Cell 3: Build training data from Qwen's own reflections
# Requires: memory_bank_qwen.json uploaded to /notebooks/CogMem/results/
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || (cd /notebooks/CogMem && git pull)

import json, os
from pathlib import Path

MB_PATH = "/notebooks/CogMem/results/memory_bank_qwen.json"
if not Path(MB_PATH).exists():
    print("ERROR: Upload memory_bank_qwen.json to /notebooks/CogMem/results/")
    raise FileNotFoundError(MB_PATH)

# Run the training data builder
os.makedirs("/notebooks/CogMem/results", exist_ok=True)
!cd /notebooks/CogMem && python3 scripts/build_training_data_qwen.py results/memory_bank_qwen.json results/training_qwen.jsonl

In [ ]:
# Cell 4: Train QLoRA on Qwen2.5:3b
import json, torch, os
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
)

os.environ["TRANSFORMERS_NO_FLASH_ATTENTION"] = "1"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
JSONL_PATH = "/notebooks/CogMem/results/training_qwen.jsonl"
ADAPTER_DIR = "/notebooks/CogMem/adapters/qwen_q_weighted"

raw_data = [json.loads(l) for l in open(JSONL_PATH)]
print(f"Training samples: {len(raw_data)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    torch_dtype=torch.float16, attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    tok = tokenizer(text, truncation=True, max_length=1024, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

dataset = Dataset.from_list(raw_data).map(format_chat, remove_columns=["messages"])
print(f"Tokenized: {len(dataset)} examples")

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=ADAPTER_DIR, num_train_epochs=5,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        learning_rate=2e-5, warmup_ratio=0.1, logging_steps=10,
        save_strategy="epoch", fp16=True, optim="paged_adamw_8bit",
        report_to="none", seed=42,
    ),
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
)

# Auto-resume from checkpoint if exists
resume = os.path.exists(ADAPTER_DIR) and any(d.startswith("checkpoint") for d in os.listdir(ADAPTER_DIR))
print(f"Starting training (resume={resume})...")
trainer.train(resume_from_checkpoint=resume)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
# Cell 5: Package adapter
!tar czf /notebooks/cogmem_qwen_adapter.tar.gz -C /notebooks/CogMem adapters/qwen_q_weighted/
!ls -lh /notebooks/cogmem_qwen_adapter.tar.gz
print("Download cogmem_qwen_adapter.tar.gz")